# exp_011 稳定锚点重训实验（Y1）

本 Notebook 独立完成：数据/缓存校验 → 锚点复现 → 近期 1702 期专家 → 多随机种子 → 三级验证（开发折/影子验证/官方 Valid）→ 权重选择 → 重训模拟 → 特征块消融 → 最终测试预测 → 指标与元数据保存。

- 特征：`legacy_328`（复用 `03_cache/processed_data_v1`，因果性已由缓存清单校验）。
- 模型：LightGBM LambdaRank（exp_003 调优参数）；全历史锚点 8 轮，近期专家 16 轮。
- 分组：每个时间点为 ranking group；每时点确定性抽取至多 1200 只股票。
- 结果：全部写入 `04_results/exp_011_stable_anchor_retrain/`，**不会覆盖** `04_results/final_submission/prediction.npy`。
- 所有模型预测按指纹缓存到结果目录 `runtime_cache/`，重复运行自动复用。

内核建议：`jingge_ts`（python 3.10 + lightgbm 4.7.0）。

## 1. 环境与路径

从当前目录向上定位项目根目录，并切换到项目根目录。

In [1]:
from __future__ import annotations
import os, sys
from pathlib import Path

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "02_experiments").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录：请从项目目录或实验目录启动 Notebook。")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
os.environ["DSCR_EXP011_AUTO_ROOT"] = "1"
print("项目根目录:", PROJECT_ROOT)

项目根目录: D:\google_dl\book\友安杯


## 2. 数据契约与 RankIC 自检

`Dataset` 会校验 READY、manifest SHA-256、legacy_328 兼容性与测试评价位置数（2,042,538）。`rank_ic_self_test` 在手工构造样本上验证官方 RankIC 实现（正/负相关、与 scipy 对照、常量、NaN/Inf、并列、样本过少）。

In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "02_experiments" / "exp_011_stable_anchor_retrain" / "src"))

from dscr_exp011_lib import Dataset, rank_ic_self_test, feature_cols_for_config
import numpy as np

assert list(feature_cols_for_config("full_328")) == list(range(328)), "full_328 列序必须为 [0:328)"
print("RankIC 自检:", rank_ic_self_test())

ds = Dataset(PROJECT_ROOT / "03_cache" / "processed_data_v1", check_sha256=True)
print(ds.split_meta().to_string(index=False))
print("数据契约校验通过。")

RankIC 自检: {'identity': 0.9999999999999999, 'reverse': -0.9999999999999999, 'vs_spearmanr': 0.0, 'constant_pred': None, 'too_few': None, 'nan_filter': -1.0, 'ties': -0.9629500128629354}
split    rows  time_start  time_stop  time_points  group_sum
train 6489099         486       2918         2432    6489099
valid  982972        2918       3161          243     982972
 test 2042538        3161       3603          442    2042538
数据契约校验通过。


## 3. 完整实验流水线

执行锚点复现（目标 Valid RankIC ≈ 0.092940）、近期专家、多随机种子（42/2026/3407）、开发折权重搜索（0.25/0.30/0.35，选定 95% 阈值内最低权重）、影子验证 [2675,2918)、官方 Valid 一次性检查、重训模拟（模拟一/二）、legacy_328 内部特征块消融、最终测试预测（Train-only 与 Train+Valid 重训两个版本）与 `prediction.npy` 生成。

所有阶段均有指纹缓存，中断后重新运行会自动续跑。

In [3]:
from run_exp011 import main as run_exp011_main

result = run_exp011_main()
print("\n=== 阶段摘要 ===")
for k, v in result.items():
    print(f"{k}: {v}")

[2026-08-07 11:40:51] exp_011 主流程启动。实验目录: D:\google_dl\book\友安杯\02_experiments\exp_011_stable_anchor_retrain


[2026-08-07 11:40:51] 结果目录: D:\google_dl\book\友安杯\04_results\exp_011_stable_anchor_retrain | 缓存目录: D:\google_dl\book\友安杯\04_results\exp_011_stable_anchor_retrain\runtime_cache


[2026-08-07 11:40:51] 数据契约校验通过（READY / manifest SHA-256 / legacy_328 兼容 / test_mask=2,042,538）。


[2026-08-07 11:40:51] RankIC 自检: {"identity": 0.9999999999999999, "reverse": -0.9999999999999999, "vs_spearmanr": 0.0, "constant_pred": null, "too_few": null, "nan_filter": -1.0, "ties": -0.9629500128629354}


[2026-08-07 11:40:51] 阶段B：训练全历史锚点 [486,2918) r8 s42，预测官方 Valid [2918,3161)。


...

[2026-08-07 11:40:59]   重训 seed=3407 anchor cached recent cached


[2026-08-07 11:41:00] 最终预测: shape=[442, 5282] eval=2042538 sha256=4e5584016b77a8533410dddc2f3bb1c66ee75b0ff23aa92b1dfc20dfd3824863 anchor_corr=0.9352


[2026-08-07 11:41:00] 晋级判定: not_promoted


[2026-08-07 11:41:00] 主流程完成，总耗时 9.5s。



=== 阶段摘要 ===
anchor_valid: 0.092940153549703
recent_valid: 0.08965672120833612
selected_weight: 0.3
shadow_blend: 0.08562847835009614
official_blend: 0.09272706238000779
retrain_pass: False
ens_stable: False
status: not_promoted
final_strategy: train_only_anchor_recent_blend_single_seed_42
prediction_sha256: 4e5584016b77a8533410dddc2f3bb1c66ee75b0ff23aa92b1dfc20dfd3824863


## 4. 最终预测文件核验

读回 `prediction.npy`，按官方口径复核 shape、dtype、有限性、评价/非评价位置数与填充值，并输出 SHA-256。

In [4]:
import hashlib, json
import numpy as np

RESULT_DIR = PROJECT_ROOT / "04_results" / "exp_011_stable_anchor_retrain"

def sha256(path):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        while c := f.read(16 * 1024 * 1024):
            d.update(c)
    return d.hexdigest()

loaded = np.load(RESULT_DIR / "prediction.npy")
test_mask = np.zeros((442, 5282), dtype=bool)
test_mask[np.asarray(ds.common["test"]["time"], dtype=np.int32) - 3161,
          np.asarray(ds.common["test"]["stock"], dtype=np.int32)] = True

summary = {
    "prediction_path": str(RESULT_DIR / "prediction.npy"),
    "shape": list(loaded.shape),
    "dtype": str(loaded.dtype),
    "finite": bool(np.isfinite(loaded).all()),
    "evaluation_count": int(test_mask.sum()),
    "non_evaluation_count": int((~test_mask).sum()),
    "non_evaluation_all_0_5": bool(np.all(loaded[~test_mask] == 0.5)),
    "min": float(loaded.min()), "max": float(loaded.max()),
    "mean": float(loaded.mean()), "std": float(loaded.std()),
    "sha256": sha256(RESULT_DIR / "prediction.npy"),
}
for k, v in summary.items():
    print(f"{k}: {v}")

assert summary["shape"] == [442, 5282] and summary["dtype"] == "float32"
assert summary["finite"] and summary["evaluation_count"] == 2_042_538 and summary["non_evaluation_all_0_5"]

metrics = json.loads((RESULT_DIR / "metrics.json").read_text(encoding="utf-8"))
print("\n最终策略:", metrics["final_strategy"])
print("晋级状态:", metrics["promotion_status"])
print("正式提交目录是否被修改: False（本实验不写 final_submission）")
print("\nexp_011 实验完成：prediction.npy 与 prediction_report.md 均已生成。")

prediction_path: D:\google_dl\book\友安杯\04_results\exp_011_stable_anchor_retrain\prediction.npy
shape: [442, 5282]
dtype: float32
finite: True
evaluation_count: 2042538
non_evaluation_count: 292106
non_evaluation_all_0_5: True
min: 0.00020500205573625863
max: 1.0
mean: 0.5000945925712585
std: 0.27001258730888367
sha256: 4e5584016b77a8533410dddc2f3bb1c66ee75b0ff23aa92b1dfc20dfd3824863

最终策略: train_only_anchor_recent_blend_single_seed_42
晋级状态: not_promoted
正式提交目录是否被修改: False（本实验不写 final_submission）

exp_011 实验完成：prediction.npy 与 prediction_report.md 均已生成。
